# 💻 NanoGPT Lab — Codex & Claude Caliber Coding AI Training
### Fine-Tune LLaMA-3.2 (1B/3B) into an Elite Coding Model on Free Google Colab GPU

This notebook trains your AI model to write production-grade code, solve complex algorithms, debug errors, and explain system architecture just like **Claude 3.5 Sonnet** and **OpenAI Codex**.

**Hardware**: Free Google Colab **T4 GPU** (`Runtime` -> `Change runtime type` -> `T4 GPU`).
**Output**: Exports a 4-bit quantized GGUF (`Llama-3.2-1B-Coder.Q4_K_M.gguf`) to run locally on your laptop CPU at ~20 tokens/sec.

In [ ]:
# Step 1: Verify Google Colab GPU
!nvidia-smi

In [ ]:
# Step 2: Install High-Performance Fine-Tuning Stack (Unsloth + PyTorch)
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
# Step 3: Load LLaMA-3.2 Base Architecture in 4-bit Quantization
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model_name = "unsloth/Llama-3.2-1B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
print("Base model loaded successfully into GPU VRAM!")

In [ ]:
# Step 4: Attach LoRA Fine-Tuning Adapters across Attention & MLP Layers
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("LoRA Adapters initialized!")

In [ ]:
# Step 5: Load High-Quality Coding Instructions (CodeAlpaca / Python Curated Dataset)
from datasets import load_dataset, Dataset

# You can load the official CodeAlpaca-20k or custom programming tasks
try:
    raw_dataset = load_dataset("sahil2801/CodeAlpaca-20k", split="train[:1200]")
    def format_prompt(sample):
        inst = sample["instruction"]
        inp = sample.get("input", "")
        out = sample["output"]
        full_inst = f"{inst}\nInput: {inp}" if inp else inst
        formatted_text = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\nYou are an expert coding assistant of Codex & Claude caliber.<|eot_id|><|start_header_id|>user<|end_header_id|>\n{full_inst}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n{out}<|eot_id|>"
        return {"text": formatted_text}
    dataset = raw_dataset.map(format_prompt)
    print("Loaded", len(dataset), "high-grade coding instructions from CodeAlpaca!")
except Exception as e:
    print("Fallback synthetic coding dataset:", e)
    code_data = [
        {"text": "<|begin_of_text|><|start_header_id|>user<|end_header_id|>\nWrite a binary search algorithm in Python.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n```python\ndef binary_search(arr, target):\n    low, high = 0, len(arr) - 1\n    while low <= high:\n        mid = (low + high) // 2\n        if arr[mid] == target:\n            return mid\n        elif arr[mid] < target:\n            low = mid + 1\n        else:\n            high = mid - 1\n    return -1\n```"}
    ]
    dataset = Dataset.from_list(code_data)

In [ ]:
# Step 6: Train the Model with SFTTrainer
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 5,
        output_dir = "outputs_coder",
    ),
)

trainer.train()
print("Fine-tuning complete! Your Coding AI is ready!")

In [ ]:
# Step 7: Export to GGUF format for Laptop CPU execution
model.save_pretrained_gguf("Llama-3.2-1B-Coder-Q4_K_M", tokenizer, quantization_method = "q4_k_m")
print("GGUF model exported! Download and place into your NanoGPT Lab root directory.")